# E3 — Esercitazione: automatizzare un report (SOLUZIONI)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrealorenzon/SIMA2026/blob/soluzioni/E3_automazione/E3_soluzioni.ipynb)

## Come funziona un'esercitazione

Rispetto alle lezioni, qui hai **meno indicazioni**: per ogni passo trovi l'obiettivo e un **controllo** per verificare di esserci arrivato. Cosa cercare, sta a te (Google, Gemini, le dispense, i notebook delle lezioni).

- 👥 Si lavora in **gruppi da 3**, con tre ruoli che **ruotano a ogni passo**: chi **scrive** (condivide lo schermo e scrive il codice), chi **cerca** (Google, Gemini, dispense), chi **controlla** (il risultato ha senso? il numero torna?).
- Se il controllo non torna, **non andare avanti**: un errore all'inizio si porta dietro tutto il resto.
- **File → Salva una copia in Drive** per non perdere il lavoro.

## Contesto

Ogni regione vuole il **suo** report: un file Excel con il fatturato mediano per anno e per settore, e un grafico. Sono 20 regioni. Farlo a mano richiederebbe un pomeriggio; con una funzione e un ciclo (L6), qualche minuto. E quando arriveranno i dati del prossimo anno, basterà rilanciare.

In [ ]:
!git clone -q --depth 1 https://github.com/andrealorenzon/SIMA2026.git

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

cartella = Path("SIMA2026/dati/per_regione")
uscita = Path("report")
uscita.mkdir(exist_ok=True)

### 1. Un report, a mano

Per la **Toscana**: costruisci una tabella con gli anni sulle righe, i settori sulle colonne e il fatturato mediano nelle celle.

✅ *Controllo:* 5 righe (anni).

In [ ]:
d = pd.read_excel(cartella / "Toscana.xlsx")
d.pivot_table(index="anno", columns="settore", values="fatturato", aggfunc="median").round(0)

### 2. La funzione

Scrivi una funzione `report(percorso)` che, per un file regionale:

1. legge il file (attenzione: i file non sono tutti identici, ricordi la L6?);
2. salva la tabella del passo 1 in `report/<regione>.xlsx`;
3. salva in `report/<regione>.png` un grafico a linea del fatturato mediano per anno, con titolo;
4. restituisce il numero di imprese della regione.

Provala sulla Toscana.

✅ *Controllo:* nel pannello dei file compaiono `report/Toscana.xlsx` e `report/Toscana.png`.

In [ ]:
def report(percorso):
    nome = Path(percorso).stem
    d = pd.read_excel(percorso)
    d.columns = d.columns.str.lower()
    tab = d.pivot_table(index="anno", columns="settore", values="fatturato", aggfunc="median")
    tab.round(1).to_excel(uscita / f"{nome}.xlsx")
    d.groupby("anno")["fatturato"].median().plot(marker="o")
    plt.title(f"Fatturato mediano, {nome}")
    plt.ylabel("Migliaia di euro")
    plt.savefig(uscita / f"{nome}.png", dpi=120, bbox_inches="tight")
    plt.close()
    return d["id_impresa"].nunique()

report(cartella / "Toscana.xlsx")

### 3. Tutte le regioni

Applica la funzione a tutti i file della cartella. Raccogli il numero di imprese di ogni regione in una tabella.

✅ *Controllo:* 40 file nella cartella `report` (20 Excel e 20 immagini) e 500 imprese in totale.

In [ ]:
righe = []
for f in sorted(cartella.glob("*.xlsx")):
    righe.append({"regione": f.stem, "imprese": report(f)})
conteggi = pd.DataFrame(righe)
print(len(list(uscita.iterdir())), conteggi["imprese"].sum())

### 4. Controlla un report a caso

Apri uno dei grafici e uno dei file Excel creati (doppio clic nel pannello dei file, o caricali con Python). Hanno senso? C'è qualche regione in cui il report è poco affidabile?

In [ ]:
conteggi.sort_values("imprese").head()

*Risposta:* Le regioni con pochissime imprese (Basilicata 2, Molise 4) hanno report con molte celle vuote e linee che saltano: con così pochi dati il report andrebbe accompagnato da un avviso. Automatizzare non esonera dal guardare i risultati.

---
## ⭐ Bonus

### B1. Un solo file

Invece di 20 file Excel, crea un unico `report/tutte.xlsx` con **un foglio per regione**.

In [ ]:
with pd.ExcelWriter(uscita / "tutte.xlsx") as w:
    for f in sorted(cartella.glob("*.xlsx")):
        d = pd.read_excel(f)
        d.columns = d.columns.str.lower()
        tab = d.pivot_table(index="anno", columns="settore", values="fatturato", aggfunc="median")
        tab.round(1).to_excel(w, sheet_name=f.stem[:31])
print("ok")

### B2. Uno zip da spedire

Comprimi la cartella `report` in un unico file da scaricare: cerca `shutil make_archive`.

In [ ]:
import shutil
shutil.make_archive("report", "zip", uscita)